# Generic benchmark hierarchical hybrid retrieval (Colab L4)

This notebook runs only the light pipeline: raw PDF/image → PDF-inspector text + rendered visual pages → BM25/V-SPLADE → Kf=3 file selection → page ranking → top-20 artifact. It does not run KDL, ColVec, baseline legacy, reranking, OCR, or QA.

Artifacts and checkpoints live on Google Drive so rerunning the same `RUN_ID` resumes completed stages.

In [ ]:
from pathlib import Path
import os
import subprocess
from google.colab import drive

drive.mount('/content/drive')

REPO_URL = 'https://github.com/iSE-UET-VNU/AXIOM_DE-RD.git'
BRANCH = 'benchmark-hierarchical'
CODE_ROOT = Path('/content/AXIOM_DE-RD-benchmark')
DRIVE_ROOT = Path('/content/drive/MyDrive/AXIOM_DE-RD')
DATASET_ROOT = DRIVE_ROOT / 'data/raw/BENCHMARK'
OUTPUT_ROOT = DRIVE_ROOT / 'data/work/benchmark_hierarchical'
RUN_ID = 'benchmark_hierarchical_kf3_metric10_save20'
RUN_DIR = OUTPUT_ROOT / RUN_ID
MODEL = 'naver/v-splade-efficient'
os.environ.setdefault('HF_HOME', '/content/huggingface-cache')
os.environ.setdefault('TRANSFORMERS_CACHE', '/content/huggingface-cache')
print({'dataset': str(DATASET_ROOT), 'output': str(RUN_DIR), 'branch': BRANCH})

In [ ]:
if not CODE_ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(CODE_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(CODE_ROOT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(CODE_ROOT), 'switch', BRANCH], check=True)
    subprocess.run(['git', '-C', str(CODE_ROOT), 'pull', '--ff-only', 'origin', BRANCH], check=True)
print(subprocess.check_output(['git', '-C', str(CODE_ROOT), 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
%pip install -q pdf-inspector PyMuPDF sentence-transformers==5.6.1 transformers==5.3.0 peft==0.20.0 fastparquet pytrec_eval-terrier
import torch
assert torch.cuda.is_available(), 'A CUDA runtime is required for the V-SPLADE stage.'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['nvidia-smi'], check=False)
assert DATASET_ROOT.is_dir(), f'Missing dataset root: {DATASET_ROOT}'
assert (DATASET_ROOT / 'documents.jsonl').is_file()
assert (DATASET_ROOT / 'queries.jsonl').is_file()
assert (DATASET_ROOT / 'qrels.jsonl').is_file()
print('Input files are present; Drive free space:')
subprocess.run(['df', '-h', str(DRIVE_ROOT)], check=False)

## Smoke test

The smoke run uses one deterministic document and two queries, disables qrels, executes every stage, and runs the same command again to verify resume. It writes to a separate Drive directory.

In [ ]:
RUNNER = CODE_ROOT / 'research/experiments/run_benchmark_hierarchical.py'
SMOKE_DIR = OUTPUT_ROOT / 'smoke'
smoke_cmd = [
    'python', '-u', str(RUNNER), '--dataset-root', str(DATASET_ROOT),
    '--output-dir', str(SMOKE_DIR), '--model', MODEL, '--device', 'cuda',
    '--file-k', '3', '--metric-page-k', '10', '--saved-page-k', '20',
    '--limit-documents', '1', '--limit-queries', '2', '--no-qrels', '--stage', 'all',
]
subprocess.run(smoke_cmd, cwd=CODE_ROOT, check=True)
subprocess.run(smoke_cmd, cwd=CODE_ROOT, check=True)
assert (SMOKE_DIR / 'runs/hierarchical_kf3_top20.jsonl').is_file()
assert (SMOKE_DIR / 'bundle/smoke.zip').is_file()
print('Smoke and resume checks passed:', SMOKE_DIR)

## Full benchmark run

The full run uses all 101 files, 5,334 pages and 220 queries from `data/raw/BENCHMARK`. Rendered images are kept in memory only; per-document sparse checkpoints, parsing cache, indexes, logs and the final bundle are persisted on Drive.

In [ ]:
full_cmd = [
    'python', '-u', str(RUNNER), '--dataset-root', str(DATASET_ROOT),
    '--output-dir', str(RUN_DIR), '--model', MODEL, '--device', 'cuda',
    '--batch-size', '3', '--query-batch-size', '64', '--render-dpi', '144',
    '--file-k', '3', '--metric-page-k', '10', '--saved-page-k', '20', '--stage', 'all',
]
subprocess.run(full_cmd, cwd=CODE_ROOT, check=True)

In [ ]:
import json
report = json.loads((RUN_DIR / 'reports/report.json').read_text())
manifest = json.loads((RUN_DIR / 'manifest.json').read_text())
run_rows = [json.loads(line) for line in (RUN_DIR / 'runs/hierarchical_kf3_top20.jsonl').read_text().splitlines() if line.strip()]
assert manifest['documents'] == 101, manifest
assert manifest['pages'] == 5334, manifest
assert manifest['queries'] == 220, manifest
assert len(run_rows) == 220
for row in run_rows:
    assert len(row['chunks']) <= 20
    assert [item['page_id'] for item in row['chunks'][:10]] == [item['page_id'] for item in row['chunks'][:20]][:10]
print(json.dumps(report.get('metrics', report), indent=2, ensure_ascii=False))
print('Report:', RUN_DIR / 'reports/report.md')
print('Top-20 run:', RUN_DIR / 'runs/hierarchical_kf3_top20.jsonl')
print('Bundle:', RUN_DIR / 'bundle' / f'{RUN_DIR.name}.zip')